# Random Forests — Implementation from Scratch
# 랜덤 포레스트 — 처음부터 구현하기

**Paper**: Leo Breiman, "Random Forests", *Machine Learning*, 45, 5–32, 2001

이 노트북은 Breiman (2001)의 Random Forest 알고리즘을 단계별로 구현합니다.
NumPy만으로 처음부터 구현한 후, scikit-learn과 비교합니다.

This notebook implements Breiman's (2001) Random Forest algorithm step by step.
We build from scratch using only NumPy, then compare with scikit-learn.

## 목차 / Table of Contents

1. **Part 1**: Decision Tree from Scratch / 의사결정 트리 직접 구현
2. **Part 2**: Bootstrap Sampling & Bagging / 부트스트랩 샘플링과 배깅
3. **Part 3**: Random Forest from Scratch / 랜덤 포레스트 직접 구현
4. **Part 4**: Out-of-Bag (OOB) Error Estimation / OOB 오류 추정
5. **Part 5**: Strength, Correlation, and c/s2 Ratio / 강도, 상관관계, c/s2 비율
6. **Part 6**: Effect of F on Strength & Correlation / F가 강도와 상관관계에 미치는 영향
7. **Part 7**: Variable Importance via Permutation / 순열 기반 변수 중요도
8. **Part 8**: Noise Robustness — RF vs AdaBoost / Noise 강건성 비교
9. **Part 9**: Comparison with scikit-learn / scikit-learn과 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

---
## Part 1: Decision Tree from Scratch
## 파트 1: 의사결정 트리 직접 구현

Random Forest의 기본 구성요소인 CART-style decision tree를 구현합니다.
각 노드에서 **Gini impurity**를 사용하여 최적 분할을 찾습니다.

We implement the CART-style decision tree, the building block of Random Forest.
Each node uses **Gini impurity** to find the best split.

### Gini Impurity / 지니 불순도

$$\text{Gini}(t) = 1 - \sum_{k=1}^{C} p_k^2$$

- $p_k$: 노드 $t$에서 클래스 $k$의 비율 / proportion of class $k$ at node $t$
- Gini = 0: 완벽히 순수 / perfectly pure
- Gini = 0.5: 최대 불순도 (2클래스) / maximum impurity (2-class)

In [ ]:
class DecisionNode:
    """A single node in a decision tree."""

    def __init__(self, feature_idx=None, threshold=None,
                 left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # Leaf node class label


def gini_impurity(y):
    """Compute Gini impurity for a set of labels.

    Gini(t) = 1 - sum(p_k^2) where p_k is the fraction of class k.
    """
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y)
    probs = counts / len(y)
    return 1.0 - np.sum(probs ** 2)


def best_split(X, y, feature_indices):
    """Find the best split among the given feature indices.

    Args:
        X: Feature matrix (n_samples, n_features).
        y: Label vector (n_samples,).
        feature_indices: Indices of features to consider.

    Returns:
        Tuple of (best_feature, best_threshold, best_gini).
    """
    best_gini = float('inf')
    best_feature = None
    best_threshold = None
    n = len(y)

    for feat_idx in feature_indices:
        thresholds = np.unique(X[:, feat_idx])
        for thresh in thresholds:
            left_mask = X[:, feat_idx] <= thresh
            right_mask = ~left_mask

            if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                continue

            gini_left = gini_impurity(y[left_mask])
            gini_right = gini_impurity(y[right_mask])

            # Weighted average Gini
            n_left = np.sum(left_mask)
            n_right = np.sum(right_mask)
            weighted_gini = (n_left * gini_left + n_right * gini_right) / n

            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feat_idx
                best_threshold = thresh

    return best_feature, best_threshold, best_gini


def build_tree(X, y, max_features=None, min_samples_leaf=1, depth=0,
               max_depth=None):
    """Recursively build a decision tree using CART with random features.

    Args:
        X: Feature matrix.
        y: Label vector.
        max_features: Number of random features to consider at each split
            (None = use all features, i.e., standard CART).
        min_samples_leaf: Minimum samples to allow a split.
        depth: Current depth.
        max_depth: Maximum allowed depth (None = unlimited).

    Returns:
        Root DecisionNode of the built tree.
    """
    n_samples, n_features = X.shape

    # Stopping conditions: pure node, too few samples, or max depth reached
    if (len(np.unique(y)) == 1 or n_samples < 2 * min_samples_leaf
            or (max_depth is not None and depth >= max_depth)):
        leaf_value = Counter(y).most_common(1)[0][0]
        return DecisionNode(value=leaf_value)

    # Random feature selection (key to Random Forest!)
    if max_features is not None and max_features < n_features:
        feature_indices = np.random.choice(
            n_features, max_features, replace=False
        )
    else:
        feature_indices = np.arange(n_features)

    feat_idx, threshold, _ = best_split(X, y, feature_indices)

    if feat_idx is None:
        leaf_value = Counter(y).most_common(1)[0][0]
        return DecisionNode(value=leaf_value)

    left_mask = X[:, feat_idx] <= threshold
    right_mask = ~left_mask

    left_child = build_tree(
        X[left_mask], y[left_mask], max_features,
        min_samples_leaf, depth + 1, max_depth
    )
    right_child = build_tree(
        X[right_mask], y[right_mask], max_features,
        min_samples_leaf, depth + 1, max_depth
    )

    return DecisionNode(
        feature_idx=feat_idx, threshold=threshold,
        left=left_child, right=right_child
    )


def predict_single(node, x):
    """Predict the class for a single sample."""
    if node.value is not None:
        return node.value
    if x[node.feature_idx] <= node.threshold:
        return predict_single(node.left, x)
    return predict_single(node.right, x)


def predict_tree(node, X):
    """Predict classes for a matrix of samples."""
    return np.array([predict_single(node, x) for x in X])


# --- Demo: single decision tree ---
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X_all, y_all = data.data, data.target

# Train/test split
n_train = int(0.8 * len(X_all))
indices = np.random.permutation(len(X_all))
X_train, y_train = X_all[indices[:n_train]], y_all[indices[:n_train]]
X_test, y_test = X_all[indices[n_train:]], y_all[indices[n_train:]]

tree = build_tree(X_train, y_train)
preds = predict_tree(tree, X_test)
single_tree_acc = np.mean(preds == y_test)
print(f"Single Decision Tree Accuracy: {single_tree_acc:.4f}")
print(f"Single Decision Tree Error:    {1 - single_tree_acc:.4f}")

---
## Part 2: Bootstrap Sampling & Bagging
## 파트 2: 부트스트랩 샘플링과 배깅

Bootstrap sampling은 Random Forest의 두 번째 핵심 요소입니다.
$N$개의 데이터에서 $N$번 복원 추출하면 약 63.2%가 포함되고, 나머지 36.8%가 OOB 데이터가 됩니다.

Bootstrap sampling is the second key component of Random Forest.
Drawing $N$ samples with replacement from $N$ data points includes ~63.2%;
the remaining ~36.8% becomes OOB data.

### Bagging의 분산 감소 원리 / Variance Reduction in Bagging

상관관계 $\rho$인 $n$개 변수의 평균의 분산:
$$\text{Var}(\bar{X}) = \rho\sigma^2 + \frac{1-\rho}{n}\sigma^2$$

- $\rho = 0$이면: $\sigma^2/n$ (이상적, 완전 독립)
- $\rho > 0$이면: $\rho\sigma^2$ 이하로 줄지 않음 → Random Forest가 $\rho$를 줄이는 이유

In [ ]:
def bootstrap_sample(X, y):
    """Create a bootstrap sample and identify OOB indices.

    Returns:
        X_boot, y_boot: Bootstrap sample.
        oob_indices: Indices of samples NOT in the bootstrap.
    """
    n = len(y)
    boot_indices = np.random.choice(n, size=n, replace=True)
    oob_indices = np.array(list(set(range(n)) - set(boot_indices)))
    return X[boot_indices], y[boot_indices], oob_indices


# Verify the ~63.2% inclusion rate
n_experiments = 1000
n_samples = 500
inclusion_rates = []

for _ in range(n_experiments):
    indices = np.random.choice(n_samples, size=n_samples, replace=True)
    unique_count = len(set(indices))
    inclusion_rates.append(unique_count / n_samples)

print(f"Average inclusion rate: {np.mean(inclusion_rates):.4f}")
print(f"Theoretical value (1 - 1/e): {1 - np.exp(-1):.4f}")
print(f"Average OOB fraction: {1 - np.mean(inclusion_rates):.4f}")

# Bagging: ensemble of trees WITHOUT random feature selection
n_bagging_trees = 50
bagging_trees = []
for _ in range(n_bagging_trees):
    X_boot, y_boot, _ = bootstrap_sample(X_train, y_train)
    tree = build_tree(X_boot, y_boot, max_features=None)  # All features
    bagging_trees.append(tree)

# Bagging prediction: majority vote
all_preds = np.array([predict_tree(t, X_test) for t in bagging_trees])
bagging_preds = np.array(
    [Counter(all_preds[:, i]).most_common(1)[0][0]
     for i in range(len(X_test))]
)
bagging_acc = np.mean(bagging_preds == y_test)
print(f"\nBagging ({n_bagging_trees} trees) Accuracy: {bagging_acc:.4f}")
print(f"Improvement over single tree: "
      f"{bagging_acc - single_tree_acc:+.4f}")

---
## Part 3: Random Forest from Scratch
## 파트 3: 랜덤 포레스트 직접 구현

Bagging에 **랜덤 feature 선택**을 추가하면 Random Forest가 됩니다.
각 노드에서 전체 $M$개 변수 대신 $F$개만 랜덤으로 후보 삼아 분할합니다.

Adding **random feature selection** to bagging creates a Random Forest.
At each node, only $F$ out of $M$ total variables are randomly considered.

Breiman의 실험 설정: / Breiman's experimental settings:
- $F = 1$: 극도로 랜덤 / extremely random
- $F = \lfloor\log_2 M + 1\rfloor$: 표준 설정 / standard setting
- 현대 관행: $F = \lfloor\sqrt{M}\rfloor$ (분류), $F = M/3$ (회귀) / Modern practice

In [ ]:
class RandomForestScratch:
    """Random Forest classifier built from scratch.

    Implements Forest-RI (Random Input) as described in Breiman (2001).
    Combines bagging with random feature selection at each node.
    """

    def __init__(self, n_trees=100, max_features='sqrt',
                 min_samples_leaf=1, max_depth=None):
        self.n_trees = n_trees
        self.max_features = max_features
        self.min_samples_leaf = min_samples_leaf
        self.max_depth = max_depth
        self.trees = []
        self.oob_indices_list = []

    def _get_max_features(self, n_features):
        """Resolve max_features parameter to an integer."""
        if self.max_features == 'sqrt':
            return int(np.sqrt(n_features))
        elif self.max_features == 'log2':
            return int(np.log2(n_features) + 1)
        elif isinstance(self.max_features, int):
            return self.max_features
        return n_features

    def fit(self, X, y):
        """Train the Random Forest."""
        self.n_classes = len(np.unique(y))
        self.n_features = X.shape[1]
        max_feat = self._get_max_features(self.n_features)

        self.trees = []
        self.oob_indices_list = []

        for _ in range(self.n_trees):
            X_boot, y_boot, oob_idx = bootstrap_sample(X, y)
            tree = build_tree(
                X_boot, y_boot,
                max_features=max_feat,
                min_samples_leaf=self.min_samples_leaf,
                max_depth=self.max_depth
            )
            self.trees.append(tree)
            self.oob_indices_list.append(oob_idx)

        self.X_train = X
        self.y_train = y
        return self

    def predict(self, X):
        """Predict classes via majority vote."""
        all_preds = np.array([predict_tree(t, X) for t in self.trees])
        # Majority vote for each sample
        predictions = np.array([
            Counter(all_preds[:, i]).most_common(1)[0][0]
            for i in range(X.shape[0])
        ])
        return predictions

    def predict_proba(self, X):
        """Predict class vote proportions."""
        all_preds = np.array([predict_tree(t, X) for t in self.trees])
        proba = np.zeros((X.shape[0], self.n_classes))
        for i in range(X.shape[0]):
            counts = np.bincount(all_preds[:, i], minlength=self.n_classes)
            proba[i] = counts / self.n_trees
        return proba


# Train Random Forest
rf = RandomForestScratch(n_trees=100, max_features='sqrt')
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_acc = np.mean(rf_preds == y_test)

print(f"Single Tree Accuracy:    {single_tree_acc:.4f}")
print(f"Bagging Accuracy:        {bagging_acc:.4f}")
print(f"Random Forest Accuracy:  {rf_acc:.4f}")
print(f"\nRF improvement over single tree: {rf_acc - single_tree_acc:+.4f}")
print(f"RF improvement over bagging:     {rf_acc - bagging_acc:+.4f}")

---
## Part 4: Out-of-Bag (OOB) Error Estimation
## 파트 4: Out-of-Bag (OOB) 오류 추정

OOB 추정은 Random Forest의 가장 실용적인 특성 중 하나입니다.
각 트리의 bootstrap에 포함되지 않은 ~36.8% 데이터를 "무료 test set"으로 사용합니다.

OOB estimation is one of the most practical features of Random Forest.
The ~36.8% of data not included in each tree's bootstrap serves as a "free test set."

Breiman (1996b)이 보인 바: OOB 오류 추정은 동일 크기의 test set을 사용한 것만큼 정확합니다.

As shown by Breiman (1996b): OOB error estimation is as accurate as using a test set of the same size.

In [ ]:
def compute_oob_error(rf):
    """Compute out-of-bag error for a trained Random Forest.

    For each training sample, aggregate votes only from trees whose
    bootstrap did NOT include that sample.
    """
    n_samples = len(rf.y_train)
    n_classes = rf.n_classes
    oob_votes = np.zeros((n_samples, n_classes))
    oob_count = np.zeros(n_samples)  # How many trees voted for this sample

    for tree, oob_idx in zip(rf.trees, rf.oob_indices_list):
        if len(oob_idx) == 0:
            continue
        preds = predict_tree(tree, rf.X_train[oob_idx])
        for j, idx in enumerate(oob_idx):
            oob_votes[idx, preds[j]] += 1
            oob_count[idx] += 1

    # Only consider samples that were OOB at least once
    valid = oob_count > 0
    oob_preds = np.argmax(oob_votes[valid], axis=1)
    oob_error = 1.0 - np.mean(oob_preds == rf.y_train[valid])

    return oob_error, np.sum(valid)


oob_error, n_valid = compute_oob_error(rf)
test_error = 1.0 - rf_acc

print(f"OOB Error:      {oob_error:.4f} (based on {n_valid} samples)")
print(f"Test Set Error: {test_error:.4f}")
print(f"Difference:     {abs(oob_error - test_error):.4f}")
print(f"\nOOB closely tracks test error — no separate test set needed!")

---
## Part 5: Strength, Correlation, and c/s2 Ratio
## 파트 5: 강도, 상관관계, c/s2 비율

Breiman의 Theorem 2.3의 핵심 양들을 OOB 데이터로 추정합니다.

We estimate the key quantities from Breiman's Theorem 2.3 using OOB data.

### 핵심 수식 / Key Equations

**Margin**: $mr(\mathbf{X}, Y) = P_\Theta(h = Y) - \max_{j \neq Y} P_\Theta(h = j)$

**Strength**: $s = E_{\mathbf{X},Y}[mr(\mathbf{X}, Y)]$

**Upper bound**: $PE^* \leq \bar{\rho}(1 - s^2) / s^2$

**c/s2 ratio**: $c/s2 = \bar{\rho} / s^2$ (작을수록 좋음 / smaller is better)

In [ ]:
def compute_strength_correlation(rf):
    """Estimate strength, correlation, and c/s2 ratio from OOB data.

    Follows Appendix II of Breiman (2001).
    """
    n_samples = len(rf.y_train)
    n_classes = rf.n_classes

    # Collect OOB vote proportions Q(x, j) for each sample
    oob_votes = np.zeros((n_samples, n_classes))
    oob_count = np.zeros(n_samples)

    for tree, oob_idx in zip(rf.trees, rf.oob_indices_list):
        if len(oob_idx) == 0:
            continue
        preds = predict_tree(tree, rf.X_train[oob_idx])
        for j, idx in enumerate(oob_idx):
            oob_votes[idx, preds[j]] += 1
            oob_count[idx] += 1

    valid = oob_count > 0
    Q = oob_votes[valid] / oob_count[valid, np.newaxis]  # Vote proportions
    y_valid = rf.y_train[valid]
    n_valid = np.sum(valid)

    # Compute margin for each sample:
    # mr(x, y) = Q(x, y) - max_{j != y} Q(x, j)
    margins = np.zeros(n_valid)
    for i in range(n_valid):
        correct_vote = Q[i, y_valid[i]]
        other_votes = np.copy(Q[i])
        other_votes[y_valid[i]] = -1  # Exclude correct class
        max_wrong_vote = np.max(other_votes)
        margins[i] = correct_vote - max_wrong_vote

    # Strength = E[margin]
    strength = np.mean(margins)

    # Variance of margin
    var_mr = np.var(margins)

    # Estimate sd(Theta) using p1, p2 (Eq. A2 in Appendix II)
    # p1 = E[I(h(X, Theta) = Y)], p2 = E[I(h(X, Theta) = j_hat(X, Y))]
    p1_vals = np.array([Q[i, y_valid[i]] for i in range(n_valid)])
    j_hat = np.zeros(n_valid, dtype=int)
    for i in range(n_valid):
        other_votes = np.copy(Q[i])
        other_votes[y_valid[i]] = -1
        j_hat[i] = np.argmax(other_votes)
    p2_vals = np.array([Q[i, j_hat[i]] for i in range(n_valid)])

    p1 = np.mean(p1_vals)
    p2 = np.mean(p2_vals)

    # sd(Theta) = sqrt(p1 + p2 + (p1 - p2)^2)  (Eq. A2)
    sd_theta = np.sqrt(p1 + p2 + (p1 - p2) ** 2)

    # rho_bar = var(mr) / (E[sd(Theta)])^2  (from Eq. 7)
    if sd_theta > 0:
        rho_bar = var_mr / (sd_theta ** 2)
    else:
        rho_bar = 0.0

    # c/s2 ratio
    cs2 = rho_bar / (strength ** 2) if strength > 0 else float('inf')

    return {
        'strength': strength,
        'correlation': rho_bar,
        'cs2': cs2,
        'var_margin': var_mr,
        'upper_bound': rho_bar * (1 - strength**2) / (strength**2)
                       if strength > 0 else float('inf')
    }


stats = compute_strength_correlation(rf)
print("=== Random Forest Statistics ===")
print(f"Strength (s):      {stats['strength']:.4f}")
print(f"Correlation (ρ̄):   {stats['correlation']:.4f}")
print(f"c/s2 ratio:        {stats['cs2']:.4f}")
print(f"PE* upper bound:   {stats['upper_bound']:.4f}")
print(f"Actual OOB error:  {oob_error:.4f}")
print(f"\nNote: PE* ≤ ρ̄(1-s²)/s² = {stats['upper_bound']:.4f} "
      f"≥ {oob_error:.4f} ✓")

---
## Part 6: Effect of F on Strength & Correlation
## 파트 6: F가 강도와 상관관계에 미치는 영향

논문의 Figure 1–3을 재현합니다. $F$를 변화시키면서 strength와 correlation,
그리고 OOB 오류를 측정합니다.

We reproduce Figures 1–3 of the paper. Varying $F$ while measuring strength,
correlation, and OOB error.

**핵심 예측 / Key prediction from theory**: $F$가 증가하면
strength와 correlation이 모두 증가하지만, strength는 빠르게 포화됩니다.
따라서 $c/s2 = \bar{\rho}/s^2$는 처음에 감소했다가 증가합니다.

In [ ]:
# Use breast cancer dataset (30 features)
M = X_train.shape[1]  # 30 features
F_values = [1, 2, 3, 4, 5, 7, 10, 15, 20, 25, 30]

results = {'F': [], 'strength': [], 'correlation': [],
           'oob_error': [], 'cs2': []}

for F in F_values:
    rf_f = RandomForestScratch(n_trees=100, max_features=F)
    rf_f.fit(X_train, y_train)

    oob_err, _ = compute_oob_error(rf_f)
    stats_f = compute_strength_correlation(rf_f)

    results['F'].append(F)
    results['strength'].append(stats_f['strength'])
    results['correlation'].append(stats_f['correlation'])
    results['oob_error'].append(oob_err)
    results['cs2'].append(stats_f['cs2'])

    print(f"F={F:2d}  s={stats_f['strength']:.3f}  "
          f"ρ̄={stats_f['correlation']:.3f}  "
          f"c/s2={stats_f['cs2']:.3f}  OOB={oob_err:.3f}")

# Plot (reproducing paper's Figure 1 style)
fig, axes = plt.subplots(2, 1, figsize=(10, 10))

# Top: Strength and Correlation vs F
ax1 = axes[0]
ax1.plot(results['F'], results['strength'], 'o-', label='Strength (s)',
         markersize=8)
ax1.plot(results['F'], results['correlation'], 's-',
         label='Correlation (ρ̄)', markersize=8)
ax1.set_xlabel('Number of Random Features (F)')
ax1.set_ylabel('Value')
ax1.set_title('Strength and Correlation vs F\n'
              '(cf. Breiman 2001, Figure 1 top)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bottom: OOB error vs F
ax2 = axes[1]
ax2.plot(results['F'], results['oob_error'], 'o-', color='tab:red',
         label='OOB Error', markersize=8)
ax2.set_xlabel('Number of Random Features (F)')
ax2.set_ylabel('Error Rate')
ax2.set_title('OOB Error vs F\n'
              '(cf. Breiman 2001, Figure 1 bottom)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 7: Variable Importance via Permutation
## 파트 7: 순열 기반 변수 중요도

논문 §10의 permutation-based variable importance를 구현합니다.

We implement the permutation-based variable importance from §10.

### 알고리즘 / Algorithm

1. 각 트리에서 OOB 데이터의 기본 오류율을 계산합니다
2. $m$번째 변수의 값을 OOB 샘플 내에서 랜덤 셔플합니다
3. 셔플 후 오류율을 다시 계산합니다
4. 오류율 증가분 = 변수 $m$의 중요도

**직관 / Intuition**: 중요한 변수를 셔플하면 label과의 관계가 깨져 오류가 크게 증가합니다.

In [ ]:
def permutation_importance(rf, n_repeats=5):
    """Compute permutation-based variable importance using OOB data.

    For each variable, permute its values in OOB data and measure
    the increase in misclassification rate.
    """
    n_features = rf.n_features
    importance = np.zeros(n_features)
    importance_count = np.zeros(n_features)

    for tree, oob_idx in zip(rf.trees, rf.oob_indices_list):
        if len(oob_idx) < 2:
            continue

        X_oob = rf.X_train[oob_idx]
        y_oob = rf.y_train[oob_idx]

        # Baseline OOB accuracy for this tree
        base_preds = predict_tree(tree, X_oob)
        base_correct = np.mean(base_preds == y_oob)

        for m in range(n_features):
            perm_drop = 0.0
            for _ in range(n_repeats):
                X_perm = X_oob.copy()
                X_perm[:, m] = np.random.permutation(X_perm[:, m])
                perm_preds = predict_tree(tree, X_perm)
                perm_correct = np.mean(perm_preds == y_oob)
                perm_drop += (base_correct - perm_correct)
            importance[m] += perm_drop / n_repeats
            importance_count[m] += 1

    # Average across all trees
    valid = importance_count > 0
    importance[valid] /= importance_count[valid]
    return importance


# Compute variable importance
print("Computing variable importance (this may take a moment)...")
imp = permutation_importance(rf, n_repeats=3)

# Plot
sorted_idx = np.argsort(imp)[::-1]
top_n = 15

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(top_n), imp[sorted_idx[:top_n]][::-1], color='steelblue')
ax.set_yticks(range(top_n))
ax.set_yticklabels(
    [data.feature_names[i] for i in sorted_idx[:top_n]][::-1]
)
ax.set_xlabel('Mean Accuracy Decrease (Permutation Importance)')
ax.set_title('Variable Importance — Breast Cancer Dataset\n'
             '(cf. Breiman 2001, §10, Figures 4-6)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
for i in range(5):
    idx = sorted_idx[i]
    print(f"  {data.feature_names[idx]:25s}: {imp[idx]:.4f}")

---
## Part 8: Noise Robustness — RF vs AdaBoost
## 파트 8: Noise 강건성 — RF vs AdaBoost

논문 §8의 실험을 재현합니다. 라벨에 5% noise를 주입한 후
Random Forest와 AdaBoost의 오류 증가를 비교합니다.

We reproduce the experiment from §8. After injecting 5% label noise,
we compare error increases for Random Forest and AdaBoost.

**Breiman의 핵심 발견**: AdaBoost는 noise에 극적으로 취약하고,
Random Forest는 거의 영향받지 않습니다.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier


def inject_label_noise(y, noise_rate=0.05):
    """Randomly flip a fraction of labels."""
    y_noisy = y.copy()
    n_flip = int(len(y) * noise_rate)
    flip_indices = np.random.choice(len(y), n_flip, replace=False)
    classes = np.unique(y)
    for idx in flip_indices:
        other_classes = classes[classes != y[idx]]
        y_noisy[idx] = np.random.choice(other_classes)
    return y_noisy


# Experiment: 50 repetitions (following Breiman's methodology)
n_reps = 50
noise_rate = 0.05

rf_clean_errors = []
rf_noisy_errors = []
ada_clean_errors = []
ada_noisy_errors = []

for rep in range(n_reps):
    # Random split
    idx = np.random.permutation(len(X_all))
    split = int(0.9 * len(X_all))
    X_tr, y_tr = X_all[idx[:split]], y_all[idx[:split]]
    X_te, y_te = X_all[idx[split:]], y_all[idx[split:]]

    # Noisy version
    y_tr_noisy = inject_label_noise(y_tr, noise_rate)

    # Random Forest (sklearn for speed)
    rf_clf = RandomForestClassifier(n_estimators=100, random_state=rep)

    rf_clf.fit(X_tr, y_tr)
    rf_clean_errors.append(1 - rf_clf.score(X_te, y_te))

    rf_clf.fit(X_tr, y_tr_noisy)
    rf_noisy_errors.append(1 - rf_clf.score(X_te, y_te))

    # AdaBoost
    ada_clf = AdaBoostClassifier(
        n_estimators=50, random_state=rep, algorithm='SAMME'
    )

    ada_clf.fit(X_tr, y_tr)
    ada_clean_errors.append(1 - ada_clf.score(X_te, y_te))

    ada_clf.fit(X_tr, y_tr_noisy)
    ada_noisy_errors.append(1 - ada_clf.score(X_te, y_te))

# Results
rf_clean_mean = np.mean(rf_clean_errors) * 100
rf_noisy_mean = np.mean(rf_noisy_errors) * 100
ada_clean_mean = np.mean(ada_clean_errors) * 100
ada_noisy_mean = np.mean(ada_noisy_errors) * 100

rf_increase = ((rf_noisy_mean - rf_clean_mean) / rf_clean_mean * 100
               if rf_clean_mean > 0 else 0)
ada_increase = ((ada_noisy_mean - ada_clean_mean) / ada_clean_mean * 100
                if ada_clean_mean > 0 else 0)

print(f"{'':20s} {'Clean':>10s} {'5% Noise':>10s} {'Increase':>10s}")
print(f"{'':20s} {'':>10s} {'':>10s} {'(%)':>10s}")
print("-" * 52)
print(f"{'Random Forest':20s} {rf_clean_mean:9.2f}% {rf_noisy_mean:9.2f}% "
      f"{rf_increase:9.1f}%")
print(f"{'AdaBoost':20s} {ada_clean_mean:9.2f}% {ada_noisy_mean:9.2f}% "
      f"{ada_increase:9.1f}%")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
x_pos = [0, 1]
width = 0.35
ax.bar([p - width/2 for p in x_pos], [rf_clean_mean, ada_clean_mean],
       width, label='Clean', color='steelblue')
ax.bar([p + width/2 for p in x_pos], [rf_noisy_mean, ada_noisy_mean],
       width, label='5% Noise', color='salmon')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Random Forest', 'AdaBoost'])
ax.set_ylabel('Test Error (%)')
ax.set_title('Noise Robustness: Random Forest vs AdaBoost\n'
             '(cf. Breiman 2001, §8, Table 4)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## Part 9: Comparison with scikit-learn
## 파트 9: scikit-learn과 비교

우리의 scratch 구현을 scikit-learn의 최적화된 구현과 비교합니다.

We compare our scratch implementation with scikit-learn's optimized version.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

# Our implementation
t0 = time.time()
rf_scratch = RandomForestScratch(n_trees=50, max_features='sqrt')
rf_scratch.fit(X_train, y_train)
scratch_time = time.time() - t0
scratch_preds = rf_scratch.predict(X_test)
scratch_acc = np.mean(scratch_preds == y_test)

# sklearn
t0 = time.time()
rf_sklearn = RandomForestClassifier(
    n_estimators=50, max_features='sqrt', random_state=42
)
rf_sklearn.fit(X_train, y_train)
sklearn_time = time.time() - t0
sklearn_acc = rf_sklearn.score(X_test, y_test)

print("=== Implementation Comparison ===")
print(f"{'':20s} {'Accuracy':>10s} {'Time (s)':>10s}")
print("-" * 42)
print(f"{'Scratch (50 trees)':20s} {scratch_acc:10.4f} "
      f"{scratch_time:10.2f}")
print(f"{'sklearn (50 trees)':20s} {sklearn_acc:10.4f} "
      f"{sklearn_time:10.2f}")
print(f"\nsklearn speedup: {scratch_time / sklearn_time:.0f}x faster")
print(f"\nNote: sklearn is heavily optimized in C/Cython.")
print(f"Our scratch implementation validates the algorithm correctly,")
print(f"achieving comparable accuracy.")

---
## 요약 / Summary

| 파트 / Part | 구현 내용 / Implementation | 핵심 발견 / Key Finding |
|---|---|---|
| 1 | Decision Tree (CART + Gini) | 단일 트리는 고분산, 불안정 / Single tree has high variance |
| 2 | Bootstrap + Bagging | 분산 감소, 하지만 트리 간 상관관계 존재 / Reduces variance, but inter-tree correlation remains |
| 3 | Random Forest (Forest-RI) | 랜덤 feature 선택이 상관관계를 낮춤 / Random feature selection reduces correlation |
| 4 | OOB Error | Test set 없이 일반화 오류 추정 가능 / Generalization error estimation without test set |
| 5 | Strength & Correlation | $PE^* \leq \bar{\rho}(1-s^2)/s^2$ 검증 / Verified Theorem 2.3 |
| 6 | Effect of F | Strength는 빠르게 포화, correlation은 계속 증가 / Strength saturates fast, correlation keeps rising |
| 7 | Variable Importance | Permutation으로 변수 기여도 측정 / Measuring variable contribution via permutation |
| 8 | Noise Robustness | RF는 noise에 강건, AdaBoost는 취약 / RF robust to noise, AdaBoost vulnerable |
| 9 | sklearn Comparison | Scratch 구현이 sklearn과 동등한 정확도 / Scratch matches sklearn accuracy |

### 다음 논문으로의 연결 / Connection to Next Papers

| 다음 논문 / Next Paper | 연결 / Connection |
|---|---|
| #12 Hinton et al. (2006) — Deep Belief Nets | 딥러닝 혁명의 시작. RF는 정형 데이터에서 여전히 경쟁력 유지 / Start of deep learning. RF remains competitive on tabular data |
| XGBoost (Chen, 2016) | RF의 앙상블 아이디어 + gradient boosting의 결합. 현재 Kaggle의 표준 / Combines RF's ensemble idea with gradient boosting. Current Kaggle standard |
| Isolation Forest | RF의 구조를 anomaly detection에 적용 / Applies RF structure to anomaly detection |